# C12-classical-models — Practice p07 — Solution


The implementation records the stable mean loss before each full-batch update and applies the mean factor once to both gradients.


In [ ]:
import numpy as np

X_p07 = np.array([[-2.0, -1.0], [-1.0, -1.5], [-0.5, 0.0],
                  [0.5, 0.2], [1.0, 1.5], [2.0, 1.0]], dtype=np.float64)
y_p07 = np.array([0.0, 0.0, 0.0, 1.0, 1.0, 1.0], dtype=np.float64)


def train_logistic(X, y, learning_rate=0.2, steps=300):
    if not isinstance(X, np.ndarray) or X.dtype != np.float64 or X.ndim != 2:
        raise ValueError("X must be a float64 matrix")
    if not isinstance(y, np.ndarray) or y.dtype != np.float64 or y.ndim != 1:
        raise ValueError("y must be a float64 vector")
    if X.shape[0] < 2 or X.shape[1] < 1 or y.shape != (X.shape[0],):
        raise ValueError("invalid X/y shapes")
    if not np.isfinite(X).all() or not np.isfinite(y).all() or not np.all((y == 0.0) | (y == 1.0)):
        raise ValueError("invalid values")
    if not np.isscalar(learning_rate) or not np.isfinite(learning_rate) or learning_rate <= 0:
        raise ValueError("learning_rate must be positive")
    if not isinstance(steps, (int, np.integer)) or isinstance(steps, bool) or steps <= 0:
        raise ValueError("steps must be a positive integer")
    w = np.zeros(X.shape[1], dtype=np.float64)
    b = 0.0
    losses = np.empty(int(steps), dtype=np.float64)
    for step in range(int(steps)):
        logits = X @ w + b
        losses[step] = np.mean(np.maximum(logits, 0.0) - y * logits + np.log1p(np.exp(-np.abs(logits))))
        probabilities = np.empty_like(logits)
        nonnegative = logits >= 0.0
        probabilities[nonnegative] = 1.0 / (1.0 + np.exp(-logits[nonnegative]))
        exp_logits = np.exp(logits[~nonnegative])
        probabilities[~nonnegative] = exp_logits / (1.0 + exp_logits)
        # PLAN018_MUTATION_TARGET: C12-p07-logistic-mean-factor
        grad_w = X.T @ (probabilities - y) / X.shape[0]
        grad_b = float(np.mean(probabilities - y))
        w -= float(learning_rate) * grad_w
        b -= float(learning_rate) * grad_b
    return {"w": w, "b": float(b), "losses": losses}


result_p07 = train_logistic(X_p07, y_p07)


### Answer check


In [ ]:
ATOL = 1e-10
RTOL = 1e-8
expected_w_p07 = np.array([3.433187300731305, 1.657175809164489])
# PLAN018_ANSWER_CHECK: C12-p07-logistic-training
assert np.allclose(result_p07["w"], expected_w_p07, atol=ATOL, rtol=RTOL)
assert np.isclose(result_p07["b"], -0.13786707666108447, atol=ATOL, rtol=RTOL)
assert np.isclose(result_p07["losses"][0], np.log(2.0), atol=ATOL, rtol=RTOL)
assert np.isclose(result_p07["losses"][-1], 0.04833379931283665, atol=ATOL, rtol=RTOL)
final_p07 = 1.0 / (1.0 + np.exp(-(X_p07 @ result_p07["w"] + result_p07["b"])))
assert np.array_equal((final_p07 >= 0.5).astype(np.int64), y_p07.astype(np.int64))
